In [14]:
import itertools
import json
import os


In [15]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "raw")


In [16]:
def load_ndjson(path: str):
	lines = []

	with open(path, "r", encoding="utf-8") as f:
		for i, line in enumerate(f):
			line = line.strip()
			if line:
				try:
					lines.append(json.loads(line))
				except json.JSONDecodeError:
					print(f"Invalid JSON in {path} (line {i}): {line[:200]}")
	return lines

def load_json(path: str):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def load_traces(dir: str):
	traces = []
	for file in os.listdir(dir):
		if file.endswith(".ndjson"):
			file_path = os.path.join(dir, file)
			content = load_ndjson(file_path)
			traces.append(content)

	# Aplanar la lista de listas devuelta por load_ndjson
	return list(itertools.chain.from_iterable(traces))


In [17]:
GAME_START_OBJ_ID = "https://w3id.org/xapi/seriousgames/activity-types/serious-game/GameStart"


In [18]:
def get_users(traces: list):
	users = set()

	for trace in traces:
		user = trace.get("actor", {}).get("account", {}).get("name")

		if user:
			obj_id = trace.get("object", {}).get("id", "")
			if obj_id == GAME_START_OBJ_ID:
				users.add(user)

	return users

In [19]:
def get_users_test(data: dict) -> set:
    users = set()

    for user in data:
        if data[user] is not None:
            users.add(user)

    return users


In [20]:
def get_users_from_code_full(folder_path: str):
	users = set()

	for file_entry in os.scandir(folder_path):
		file_name = file_entry.name

		if "code" in file_name and file_name.endswith(".json"):
			code_data = load_json(file_entry.path)
			code_users = get_users_test(code_data)
			users = users | code_users

		elif "full" in file_name and file_name.endswith(".json"):
			full_data = load_json(file_entry.path)
			full_users = get_users_test(full_data)
			users = users | full_users

	return users


In [21]:
total_users = set()

for session_entry in os.scandir(raw_dir):
	session_dir = session_entry.path

	for folder_name in ("Gameplay", "Post", "Pre"):
		dir = os.path.join(session_dir, folder_name)

		if folder_name == "Gameplay":
			traces = load_traces(dir)
			users = get_users(traces)

		if folder_name == "Post":
			users = get_users_from_code_full(dir)

		elif folder_name == "Pre":
			users = get_users_from_code_full(dir)
			
		total_users.update(users)


In [22]:
print(len(total_users))


131


In [23]:
excluded_users = {
	"memogames",
	"682b4a72c76d2e0023ed4d05_igbm",
	"682b4a41c76d2e0023ed4b24_dmyj",
	"682b4a41c76d2e0023ed4b24_sxif",
	"682b4a72c76d2e0023ed4d05_agog"
}

valid_users = total_users - excluded_users


In [24]:
print(len(valid_users))


126
